# Feature Engineering 
## Objective

In this notebook, we transform the processed EPİAŞ PTF dataset into a machine learning-ready feature set.

We will:

- Create time-based features (hour, day, seasonality)
- Add lag features (short-term & weekly memory)
- Add rolling statistics (trend & volatility)
- Engineer currency-based features (PTF/USD, PTF/EUR)
- Generate target variable for forecasting
- Prepare final dataset for modeling

This notebook follows production-level practices and avoids data leakage.

In [36]:
import pandas as pd
import numpy as np
from pathlib import Path


PROCESSED_PATH = Path("../data/processed/ptf/ptf_processed.parquet")
FEATURE_PATH = Path("../data/features/ptf/ptf_features_train.parquet")


TARGET_HORIZON = 24  

In [37]:
df = pd.read_parquet(PROCESSED_PATH)

print(df.shape)
df.head()

(72504, 9)


,date,hour,ptf,priceusd,priceeur,_chunk_start,_chunk_end,year,month
0,2018-01-01 00:00:00+03:00,00:00,207.60,55.04,45.97,2018-01-01,2018-01-07,2018,1
1,2018-01-01 01:00:00+03:00,01:00,205.34,54.44,45.47,2018-01-01,2018-01-07,2018,1
2,2018-01-01 02:00:00+03:00,02:00,164.94,43.73,36.53,2018-01-01,2018-01-07,2018,1
3,2018-01-01 03:00:00+03:00,03:00,154.52,40.97,34.22,2018-01-01,2018-01-07,2018,1
4,2018-01-01 04:00:00+03:00,04:00,112.64,29.86,24.95,2018-01-01,2018-01-07,2018,1


In [38]:
df = df.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df = df.set_index("date")
df = df[~df.index.duplicated(keep="first")]
df = df.dropna(subset=["ptf"])

df.head()

,hour,ptf,priceusd,priceeur,_chunk_start,_chunk_end,year,month
date,,,,,,,,
2018-01-01 00:00:00+03:00,00:00,207.60,55.04,45.97,2018-01-01,2018-01-07,2018,1
2018-01-01 01:00:00+03:00,01:00,205.34,54.44,45.47,2018-01-01,2018-01-07,2018,1
2018-01-01 02:00:00+03:00,02:00,164.94,43.73,36.53,2018-01-01,2018-01-07,2018,1
2018-01-01 03:00:00+03:00,03:00,154.52,40.97,34.22,2018-01-01,2018-01-07,2018,1
2018-01-01 04:00:00+03:00,04:00,112.64,29.86,24.95,2018-01-01,2018-01-07,2018,1


## Time Features

We extract temporal signals to help the model learn seasonality patterns.

- Hour of day
- Day of week
- Weekend indicator
- Cyclical encoding (sin/cos)

In [40]:
df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

In [41]:
df.head()

,hour,ptf,priceusd,priceeur,_chunk_start,_chunk_end,year,month,day_of_week,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos
date,,,,,,,,,,,,,,
2018-01-01 00:00:00+03:00,0,207.60,55.04,45.97,2018-01-01,2018-01-07,2018,1,0,0,0.000000,1.000000,0.0,1.0
2018-01-01 01:00:00+03:00,1,205.34,54.44,45.47,2018-01-01,2018-01-07,2018,1,0,0,0.258819,0.965926,0.0,1.0
2018-01-01 02:00:00+03:00,2,164.94,43.73,36.53,2018-01-01,2018-01-07,2018,1,0,0,0.500000,0.866025,0.0,1.0
2018-01-01 03:00:00+03:00,3,154.52,40.97,34.22,2018-01-01,2018-01-07,2018,1,0,0,0.707107,0.707107,0.0,1.0
2018-01-01 04:00:00+03:00,4,112.64,29.86,24.95,2018-01-01,2018-01-07,2018,1,0,0,0.866025,0.500000,0.0,1.0


## Currency-Based Features

We normalize electricity price using exchange rates:

- PTF / USD
- PTF / EUR

This helps the model understand real purchasing power and macro effects.

In [43]:

df["price_usd"] = df["priceusd"].replace(0, np.nan)
df["price_eur"] = df["priceeur"].replace(0, np.nan)

df["ptf_usd_ratio"] = df["ptf"] / df["price_usd"]
df["ptf_eur_ratio"] = df["ptf"] / df["price_eur"]

df.head()

,hour,ptf,priceusd,priceeur,_chunk_start,_chunk_end,year,month,day_of_week,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,price_usd,price_eur,ptf_usd_ratio,ptf_eur_ratio
date,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00+03:00,0,207.60,55.04,45.97,2018-01-01,2018-01-07,2018,1,0,0,0.000000,1.000000,0.0,1.0,55.04,45.97,3.771802,4.515989
2018-01-01 01:00:00+03:00,1,205.34,54.44,45.47,2018-01-01,2018-01-07,2018,1,0,0,0.258819,0.965926,0.0,1.0,54.44,45.47,3.771859,4.515945
2018-01-01 02:00:00+03:00,2,164.94,43.73,36.53,2018-01-01,2018-01-07,2018,1,0,0,0.500000,0.866025,0.0,1.0,43.73,36.53,3.771781,4.515193
2018-01-01 03:00:00+03:00,3,154.52,40.97,34.22,2018-01-01,2018-01-07,2018,1,0,0,0.707107,0.707107,0.0,1.0,40.97,34.22,3.771540,4.515488
2018-01-01 04:00:00+03:00,4,112.64,29.86,24.95,2018-01-01,2018-01-07,2018,1,0,0,0.866025,0.500000,0.0,1.0,29.86,24.95,3.772271,4.514629


## Lag Features

Lag features allow the model to learn historical dependencies.

We use:
- Short-term memory (1, 3 hours)
- Daily pattern (24 hours)
- Weekly pattern (168 hours)

In [47]:
lags = [1, 3, 24, 48, 72, 168, 336]

for lag in lags:
    df[f"lag_{lag}"] = df["ptf"].shift(lag)

df.head()

,hour,ptf,priceusd,priceeur,_chunk_start,_chunk_end,year,month,day_of_week,is_weekend,...,price_eur,ptf_usd_ratio,ptf_eur_ratio,lag_1,lag_3,lag_24,lag_48,lag_72,lag_168,lag_336
date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00+03:00,0,207.60,55.04,45.97,2018-01-01,2018-01-07,2018,1,0,0,...,45.97,3.771802,4.515989,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 01:00:00+03:00,1,205.34,54.44,45.47,2018-01-01,2018-01-07,2018,1,0,0,...,45.47,3.771859,4.515945,207.60,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 02:00:00+03:00,2,164.94,43.73,36.53,2018-01-01,2018-01-07,2018,1,0,0,...,36.53,3.771781,4.515193,205.34,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 03:00:00+03:00,3,154.52,40.97,34.22,2018-01-01,2018-01-07,2018,1,0,0,...,34.22,3.771540,4.515488,164.94,207.60,NaN,NaN,NaN,NaN,NaN
2018-01-01 04:00:00+03:00,4,112.64,29.86,24.95,2018-01-01,2018-01-07,2018,1,0,0,...,24.95,3.772271,4.514629,154.52,205.34,NaN,NaN,NaN,NaN,NaN


## Rolling Statistics

Rolling features capture trend and volatility.

We shift by 1 to prevent data leakage.

In [53]:
base = df["ptf"].shift(1)

windows = [24, 168]

for w in windows:
    df[f"rolling_mean_{w}"] = base.rolling(w).mean()
    df[f"rolling_std_{w}"] = base.rolling(w).std()
    df[f"rolling_min_{w}"] = base.rolling(w).min()
    df[f"rolling_max_{w}"] = base.rolling(w).max()

## Difference Features

Differences help capture momentum and sudden changes.

In [54]:
df["diff_1"] = df["ptf"].diff(1)
df["diff_24"] = df["ptf"].diff(24)
df["diff_168"] = df["ptf"].diff(168)

## Feature Stability — Handling Extreme Values

Electricity prices may contain extreme spikes due to market conditions.

To improve model stability, we optionally clip extreme values using quantiles.

This step is especially useful for tree-based models and neural networks.

In [75]:
q_low = df["ptf"].quantile(0.01)
q_high = df["ptf"].quantile(0.99)

print(f"Clipping range: [{q_low:.2f}, {q_high:.2f}]")

df["ptf_clipped"] = df["ptf"].clip(q_low, q_high)
df["spike_flag"] = (df["ptf"].shift(1) > q_high).astype(int)


Clipping range: [29.99, 4500.00]


## Recompute Features 

We can recompute critical features using the clipped price to reduce noise impact.

In [76]:
for lag in [24, 168]:
    df[f"lag_{lag}_clipped"] = df["ptf_clipped"].shift(lag)

base_clipped = df["ptf_clipped"].shift(1)

df["rolling_mean_24_clipped"] = base_clipped.rolling(24).mean()
df["rolling_mean_168_clipped"] = base_clipped.rolling(168).mean()

## Target Variable

We predict next day's same hour price.

In [77]:
df["target"] = df["ptf"].shift(-TARGET_HORIZON)

## Final Cleaning

Drop rows with NaNs caused by lagging and rolling.

In [78]:
df = df.dropna()

print(df.shape)

(71542, 43)


In [79]:
feature_cols = [
    "hour", "day_of_week", "is_weekend",
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
  
    "ptf_usd_ratio", "ptf_eur_ratio",

    *[f"lag_{l}" for l in [1,3,24,48,72,168,336]],
    "lag_24_clipped",
    "lag_168_clipped",
    "rolling_mean_24_clipped",
    "rolling_mean_168_clipped",

   
    *[f"rolling_mean_{w}" for w in [24,168]],
    *[f"rolling_std_{w}" for w in [24,168]],
    *[f"rolling_min_{w}" for w in [24,168]],
    *[f"rolling_max_{w}" for w in [24,168]],


    "diff_1", "diff_24", "diff_168",

    "spike_flag"
]

target_col = "target"

X = df[feature_cols]
y = df[target_col]

print(X.shape, y.shape)

(71542, 32) (71542,)


In [80]:
X.columns

Index(['hour', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'ptf_usd_ratio', 'ptf_eur_ratio', 'lag_1', 'lag_3', 'lag_24',
       'lag_48', 'lag_72', 'lag_168', 'lag_336', 'lag_24_clipped',
       'lag_168_clipped', 'rolling_mean_24_clipped',
       'rolling_mean_168_clipped', 'rolling_mean_24', 'rolling_mean_168',
       'rolling_std_24', 'rolling_std_168', 'rolling_min_24',
       'rolling_min_168', 'rolling_max_24', 'rolling_max_168', 'diff_1',
       'diff_24', 'diff_168', 'spike_flag'],
      dtype='object')